# 02 · String problems

Fifteen string questions that come up again and again in coding screens, from reversing words to a sliding-window substring search, plus the regex and unicode details that separate a working answer from a fragile one. Afterwards you can pick the right pattern (a counter, a stack, a sliding window, a lookup table) within the first minute.

**Time:** ~50 min · **Runs:** offline on CPU in < 15 s · **Needs:** [01 · Warm-up problems](01_warmup_problems.ipynb), [Strings and f-strings](../01_python_basics/02_strings_and_fstrings.ipynb)

## Why this matters in interviews

- Strings are the most common coding-screen input, and most questions reduce to four patterns: **count** (hash map), **match nesting** (stack), **scan a window** (two pointers), **map symbols** (lookup table).
- AI-engineer work is text work. Normalising, tokenising, validating and cleaning strings is what every RAG or LLM pipeline does before any model runs.
- Unicode and regex details (code points vs characters, `casefold`, `fullmatch`) are the follow-ups that show you have shipped text code.

## What interviewers ask

- "Reverse the words in a sentence" / "find the first character that doesn't repeat."
- "Check if the brackets are balanced." (a stack)
- "Longest substring without repeating characters." (the sliding-window classic)
- "Count the words and give me the top ten, ignoring stopwords."
- "Convert Roman numerals to integers and back."
- "Validate an email address." Then: "Why is regex a bad email validator?"
- "Normalise this text for search: case, accents, whitespace."
- Always: "What's the complexity? What about empty strings, unicode, repeated characters?"

Each problem has a statement, a solution cell, a test cell (`assert`s: silence means pass) and one line on complexity. To practise, write your own version and add it to the tuple the test cell loops over.

In [ ]:
import os
import random
import re
import string
import time
import unicodedata
from collections import Counter
from itertools import groupby
from urllib.parse import urlparse

import matplotlib.pyplot as plt

random.seed(0)
plt.rcParams.update({                      # quiet chart style; colour-blind-safe colours below
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#c3c2b7",
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.8,
    "axes.axisbelow": True, "axes.titlesize": 11, "legend.frameon": False,
})
BLUE, ORANGE, GREY = "#2a78d6", "#eb6834", "#c3c2b7"


def raises(exc_type, fn, *args, **kwargs) -> bool:
    """True if fn(*args) raises exc_type (a tiny stand-in for pytest.raises)."""
    try:
        fn(*args, **kwargs)
    except exc_type:
        return True
    return False


TEXT = """Retrieval-augmented generation (RAG) answers a question in two steps. First, a retriever
searches the document store for the chunks most similar to the question. Second, the model reads
those chunks and writes an answer that cites them. If retrieval misses the right chunks, the model
cannot answer well, so most RAG bugs are retrieval bugs. Good chunks are small enough to be specific
and large enough to be useful. Hybrid retrieval mixes keyword search with vector search, and a
reranker reorders the retrieved chunks before the model sees them. Evaluate retrieval and
generation separately: retrieval with recall, the answer with faithfulness to the retrieved chunks.
Don't skip evaluation; it's how you find out which step failed."""
print(f"sample text: {len(TEXT.split())} words, {len(TEXT)} characters")

## 1 · Words and whitespace

### Problem 1 — Reverse the words in a sentence

Collapse extra whitespace. Example: `"  the sky   is blue "` → `"blue is sky the"`.

In [ ]:
def reverse_words(s: str) -> str:
    return " ".join(reversed(s.split()))       # split() with no argument splits on runs of whitespace


def reverse_words_manual(s: str) -> str:
    words, current = [], []
    for ch in s + " ":                          # a trailing sentinel space flushes the last word
        if ch.isspace():
            if current:
                words.append("".join(current))
                current = []
        else:
            current.append(ch)
    return " ".join(words[::-1])


print(repr(reverse_words("  the sky   is blue ")))

In [ ]:
for f in (reverse_words, reverse_words_manual):
    assert f("") == "" and f("   ") == ""                    # empty / only spaces
    assert f("one") == "one"                                  # single word
    assert f("  the sky   is blue ") == "blue is sky the"     # extra spaces collapse
    assert f("a\tb\nc") == "c b a"                            # tabs and newlines are whitespace too
    assert f("a a b") == "b a a"                              # duplicates
    assert f("héllo wörld") == "wörld héllo"                  # unicode
assert "a  b".split(" ") == ["a", "", "b"]                    # the trap: split(" ") keeps empty strings
print("reverse_words: all tests passed")

**Complexity:** O(n) time and space. The in-place follow-up (for a mutable character array, as in C or Java) is: reverse the whole string, then reverse each word.

### Problem 2 — Title case with exceptions

Capitalise every word except small words (a, an, the, of, and, …), but always capitalise the first and last word.
Example: `"the lord of the rings"` → `"The Lord of the Rings"`.

In [ ]:
SMALL_WORDS = {"a", "an", "and", "as", "at", "but", "by", "for", "in", "nor",
               "of", "on", "or", "the", "to", "up"}


def title_case(text: str, small: set[str] = SMALL_WORDS) -> str:
    words = text.lower().split()
    last = len(words) - 1
    return " ".join(
        w if 0 < i < last and w in small else w[:1].upper() + w[1:]   # first and last always capitalised
        for i, w in enumerate(words)
    )


print(title_case("the lord of the rings"), "|", "they're bill's friends".title())

In [ ]:
assert title_case("") == ""                                   # empty
assert title_case("a") == "A"                                 # a lone small word is first AND last
assert title_case("the lord of the rings") == "The Lord of the Rings"
assert title_case("WAR AND PEACE") == "War and Peace"         # input case doesn't matter
assert title_case("what is it for") == "What Is It For"       # the last word is always capitalised
assert title_case("  extra   spaces ") == "Extra Spaces"
assert title_case("they're bill's friends") == "They're Bill's Friends"
assert "they're bill's friends".title() == "They'Re Bill'S Friends"      # why not str.title()
assert string.capwords("they're bill's friends") == "They're Bill's Friends"   # stdlib, but no exceptions
assert title_case("élan vital") == "Élan Vital"               # unicode
print("title_case: all tests passed")

**Complexity:** O(n). `str.title()` treats every non-letter as a word boundary, so apostrophes break it; `string.capwords` splits on whitespace only. Follow-ups: acronyms (`NASA`), hyphenated words, and names like `McDonald`.

### Problem 3 — Word count and top words, excluding stopwords

Lower-case, split into words (keep `don't` whole), drop stopwords, return the top k with ties broken alphabetically.
Example: `top_words("y x y x z", k=2)` → `[("x", 2), ("y", 2)]`.

In [ ]:
STOPWORDS = frozenset("""
a about after all also an and any are as at be been before being but by can cannot could did do
does don't for from had has have he her his how i if in into is it it's its just me more most my
no not of on one or our out she should so some such than that the their them then there these they
this those to too two up us was we well were what when which who why will with would you your
""".split())
WORD_RE = re.compile(r"\w+(?:'\w+)*")          # unicode letters/digits; keeps don't and it's whole


def tokenize(text: str) -> list[str]:
    return WORD_RE.findall(text.lower())


def top_words(text: str, k: int = 10, stopwords=STOPWORDS) -> list[tuple[str, int]]:
    counts = Counter(w for w in tokenize(text) if w not in stopwords)
    return sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))[:k]   # count desc, then word asc


print(top_words(TEXT, k=5))

In [ ]:
assert tokenize("") == [] and top_words("") == []                         # empty
assert top_words("the the the") == []                                     # only stopwords
assert top_words("Model, model! MODEL.") == [("model", 3)]                # case and punctuation
assert tokenize("Don't stop") == ["don't", "stop"]                        # apostrophes stay inside words
assert top_words("y x y x z", k=2) == [("x", 2), ("y", 2)]                # ties: alphabetical...
assert Counter("y x y x z".split()).most_common(2) == [("y", 2), ("x", 2)]   # ...most_common keeps first-seen order
assert top_words("café café naïve") == [("café", 2), ("naïve", 1)]        # \w is unicode-aware
assert top_words(TEXT, k=100)[-1][1] == 1 and len(top_words(TEXT, k=3)) == 3
print("top_words: all tests passed")

In [ ]:
top10 = top_words(TEXT, k=10)
words, counts = zip(*top10)
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.barh(words[::-1], counts[::-1], color=BLUE, height=0.6)
for y, c in enumerate(counts[::-1]):
    ax.text(c + 0.08, y, str(c), va="center", fontsize=9, color="#52514e")
ax.grid(axis="y", visible=False)
ax.set_xlabel("count in the sample text")
ax.set_title("Top 10 words after removing stopwords: the content words are what's left")
plt.show()

**Complexity:** O(n) to tokenise and count, plus O(m log m) to sort m distinct words (`heapq.nlargest` makes it O(m log k)). Say which tie-break you use; `most_common` breaks ties by first appearance. For real work, use a curated stopword list such as scikit-learn's `ENGLISH_STOP_WORDS`.

### Problem 4 — Normalise text for search

Case-fold, strip accents with `unicodedata`, collapse whitespace.
Example: `"  Café   CRÈME\tbrûlée "` → `"cafe creme brulee"`, so the query `"creme brulee"` finds `"Crème Brûlée"`.

**In plain English:** the same word can be stored as different code points (`é` as one character, or `e` plus a combining accent). NFKD decomposition splits accents off so you can drop them, and also unpacks look-alikes such as the `ﬁ` ligature and full-width letters.

In [ ]:
def normalise(text: str) -> str:
    decomposed = unicodedata.normalize("NFKD", text)          # é -> e + combining accent; ﬁ -> fi
    no_marks = "".join(ch for ch in decomposed if not unicodedata.combining(ch))
    return " ".join(no_marks.casefold().split())              # fold case, collapse whitespace


docs = ["Crème Brûlée recipe", "CREME   brulee\tfor two", "Café au lait", "Brûlée, the French way"]
query = "creme brulee"
print("plain `in`   :", [d for d in docs if query in d.lower()])
print("normalised in:", [d for d in docs if normalise(query) in normalise(d)])

In [ ]:
assert normalise("") == "" and normalise("  \t\n ") == ""               # empty / whitespace only
assert normalise("  Café   CRÈME\tbrûlée ") == "cafe creme brulee"
assert normalise("Straße") == "strasse"                                 # casefold: ß -> ss
assert normalise("ﬁle") == "file"                                       # NFKD unpacks the ligature
assert normalise("ＡＢＣ１２３") == "abc123"                               # full-width forms
composed, decomposed = "é", "é"                              # two encodings of é
assert composed != decomposed and normalise(composed) == normalise(decomposed) == "e"
assert normalise("йогурт") == "иогурт"                                  # the trap: й loses its breve, becomes и
assert normalise(normalise("Crème Brûlée")) == normalise("Crème Brûlée")   # idempotent
print("normalise: all tests passed")

**Complexity:** O(n). Stripping marks is lossy where the mark changes the letter (Russian й → и, Spanish ñ → n), so apply it to the search key only and keep the original text. Use NFC (not NFKD) when you only want identical strings to compare equal.

## 2 · Characters and counting

### Problem 5 — First non-repeating character

Example: `"leetcode"` → `"l"`; `"loveleetcode"` → `"v"`; `"aabb"` → `None`.

In [ ]:
def first_unique_brute(s: str):
    for ch in s:
        if s.count(ch) == 1:                  # count() rescans the string: O(n²) overall
            return ch
    return None


def first_unique(s: str):
    counts = Counter(s)                       # pass 1: count
    return next((ch for ch in s if counts[ch] == 1), None)   # pass 2: first with count 1


print(first_unique("leetcode"), first_unique("loveleetcode"), first_unique("aabb"))

In [ ]:
for f in (first_unique_brute, first_unique):
    assert f("") is None                                  # empty
    assert f("a") == "a"                                  # single
    assert f("aabb") is None                              # all repeat
    assert f("leetcode") == "l" and f("loveleetcode") == "v"
    assert f("aA") == "a"                                 # case-sensitive (clarify!)
    assert f("a a") == " "                                # whitespace is a character too
    assert f("😀😀é") == "é"                               # unicode
print("first_unique: all tests passed")

**Complexity:** O(n) time and O(k) space for k distinct characters (the brute force is O(n²)). For a *stream* of characters, keep the counts plus an ordered dict of the current unique characters.

### Problem 6 — Pangram

Does the sentence use every letter a–z at least once?
Example: `"The quick brown fox jumps over the lazy dog"` → `True`.

In [ ]:
ALPHABET = frozenset(string.ascii_lowercase)


def is_pangram(s: str) -> bool:
    return ALPHABET <= set(s.lower())                    # <= on sets means "is a subset of"


def missing_letters(s: str) -> list[str]:
    return sorted(ALPHABET - set(s.lower()))


print(is_pangram("The quick brown fox jumps over the lazy dog"), missing_letters("The quick brown fox jumps over the lay dog"))

In [ ]:
assert not is_pangram("") and missing_letters("") == list(string.ascii_lowercase)   # empty
assert not is_pangram("a")                                                          # single
assert is_pangram("The quick brown fox jumps over the lazy dog")
assert is_pangram("Pack my box with five dozen liquor jugs")
assert is_pangram(string.ascii_uppercase)                                           # case-insensitive
assert missing_letters("The quick brown fox jumps over the lay dog") == ["z"]
assert missing_letters("Äbcdefghijklmnopqrstuvwxyz") == ["a"]                       # Ä is not a
print("pangram: all tests passed")

**Complexity:** O(n) time; the alphabet set is O(1) space, and `set(s)` is O(k). An early-exit loop stops as soon as all 26 letters are seen.

### Problem 7 — Isomorphic strings

Can the characters of `a` be replaced one-to-one to get `b`?
Example: `("egg", "add")` → `True`; `("foo", "bar")` → `False`; `("badc", "baba")` → `False` (b and d can't both map to b).

In [ ]:
def is_isomorphic_brute(a: str, b: str) -> bool:
    """Same 'shape': each char replaced by the index of its first occurrence. O(n²)."""
    return len(a) == len(b) and [a.index(c) for c in a] == [b.index(c) for c in b]


def is_isomorphic(a: str, b: str) -> bool:
    if len(a) != len(b):
        return False
    a_to_b, b_to_a = {}, {}
    for x, y in zip(a, b):
        if a_to_b.setdefault(x, y) != y or b_to_a.setdefault(y, x) != x:   # both directions must agree
            return False
    return True


def is_isomorphic_sets(a: str, b: str) -> bool:
    return len(a) == len(b) and len(set(a)) == len(set(b)) == len(set(zip(a, b)))


print(is_isomorphic("egg", "add"), is_isomorphic("foo", "bar"), is_isomorphic("badc", "baba"))

In [ ]:
for f in (is_isomorphic_brute, is_isomorphic, is_isomorphic_sets):
    assert f("", "") and f("a", "b")                      # empty, single
    assert f("egg", "add") and f("paper", "title")
    assert not f("foo", "bar")                            # o would map to both a and r
    assert not f("badc", "baba")                          # b and d would both map to b
    assert not f("ab", "a")                               # different lengths
    assert f("😀😀x", "ééy")                               # unicode
for _ in range(500):                                      # property test: all three agree
    a = "".join(random.choice("abc") for _ in range(random.randint(0, 6)))
    b = "".join(random.choice("xyz") for _ in range(len(a)))
    assert is_isomorphic_brute(a, b) == is_isomorphic(a, b) == is_isomorphic_sets(a, b)
print("isomorphic: all tests passed")

**Complexity:** O(n) time, O(k) space. The trap is checking only one direction: `"badc" → "baba"` passes a one-way map. The set one-liner works because a bijection needs as many distinct pairs as distinct characters on each side.

### Problem 8 — String rotation check

Is `b` a rotation of `a`? Example: `("waterbottle", "erbottlewat")` → `True`.

**In plain English:** every rotation of `a` appears inside `a + a`, so the question becomes a substring search.

In [ ]:
def is_rotation_brute(a: str, b: str) -> bool:
    return len(a) == len(b) and any(a[i:] + a[:i] == b for i in range(len(a) or 1))   # O(n²)


def is_rotation(a: str, b: str) -> bool:
    return len(a) == len(b) and b in a + a


print(is_rotation("waterbottle", "erbottlewat"), is_rotation("abc", "acb"))

In [ ]:
for f in (is_rotation_brute, is_rotation):
    assert f("", "") and f("a", "a")                      # empty, single
    assert f("ab", "ba") and f("waterbottle", "erbottlewat")
    assert not f("abc", "acb")                            # same letters, not a rotation
    assert not f("aa", "aaa")                             # without the length check, "aaa" in "aaaa" is True!
    assert f("aab", "aba") and not f("aab", "bba")        # duplicates
    assert f("héllo", "llohé")                            # unicode
assert "aaa" in "aa" + "aa"                               # ...which is why the length check matters
print("rotation: all tests passed")

**Complexity:** O(n) time and space for `b in a + a` (CPython's substring search is linear in practice); the brute force is O(n²).

## 3 · Classic string algorithms

### Problem 9 — Longest common prefix

Example: `["flower", "flow", "flight"]` → `"fl"`; `["dog", "car"]` → `""`.

In [ ]:
def lcp_vertical(words: list[str]) -> str:
    if not words:
        return ""
    for i, ch in enumerate(words[0]):             # compare column by column
        if any(i >= len(w) or w[i] != ch for w in words[1:]):
            return words[0][:i]
    return words[0]


def lcp_zip(words: list[str]) -> str:
    prefix = []
    for column in zip(*words):                    # zip stops at the shortest word
        if len(set(column)) != 1:
            break
        prefix.append(column[0])
    return "".join(prefix)


print(lcp_vertical(["flower", "flow", "flight"]), lcp_zip(["interview", "internet", "interval"]))

In [ ]:
for f in (lcp_vertical, lcp_zip, os.path.commonprefix):   # the stdlib has it (character-wise, not path-aware)
    assert f([]) == ""                                     # empty list
    assert f(["alone"]) == "alone"                         # single word
    assert f(["", "a"]) == ""                              # an empty word
    assert f(["dog", "car"]) == ""
    assert f(["flower", "flow", "flight"]) == "fl"
    assert f(["same", "same"]) == "same"                   # duplicates
    assert f(["naïve", "naïf"]) == "naï"                   # unicode
    assert f(["Apple", "apple"]) == ""                     # case-sensitive
print("longest common prefix: all tests passed")

**Complexity:** O(S) time where S is the total number of characters, and O(1) extra space. Vertical scanning stops at the first mismatch. For many queries against a fixed word set, a trie is the follow-up.

### Problem 10 — Valid parentheses

Brackets `()[]{}` must close in the right order; ignore every other character.
Example: `"([]{})"` → `True`; `"([)]"` → `False`; `"("` → `False`.

In [ ]:
PAIRS = {")": "(", "]": "[", "}": "{"}


def is_valid_brackets_brute(s: str) -> bool:
    s = "".join(ch for ch in s if ch in "()[]{}")
    previous = None
    while s != previous:                                  # delete innermost pairs until nothing changes
        previous, s = s, s.replace("()", "").replace("[]", "").replace("{}", "")
    return s == ""


def is_valid_brackets(s: str) -> bool:
    stack = []
    for ch in s:
        if ch in "([{":
            stack.append(ch)
        elif ch in PAIRS:
            if not stack or stack.pop() != PAIRS[ch]:     # nothing to close, or the wrong kind
                return False
    return not stack                                      # leftovers are unclosed brackets


print(is_valid_brackets("([]{})"), is_valid_brackets("([)]"), is_valid_brackets("f(x[0]) + {y}"))

In [ ]:
for f in (is_valid_brackets_brute, is_valid_brackets):
    assert f("") and f("abc")                              # empty, no brackets at all
    assert not f("(") and not f(")")                       # single
    assert f("()") and f("([]{})") and f("{[()()]}")
    assert not f("([)]") and not f("((()") and not f("}{")
    assert f("a(b)c")                                      # other characters are ignored
for _ in range(2000):                                      # property test on random bracket strings
    s = "".join(random.choice("()[]{}") for _ in range(random.randint(0, 8)))
    assert is_valid_brackets(s) == is_valid_brackets_brute(s), s
print("valid brackets: all tests passed")

**Complexity:** O(n) time, O(n) space for the stack (the replace loop is O(n²)). With a single bracket type a counter is enough, but a counter can't catch `"([)]"`, which is why the stack matters.

### Problem 11 — Longest substring without repeating characters

Example: `"abcabcbb"` → `3` (`"abc"`); `"bbbbb"` → `1`; `"pwwkew"` → `3` (`"wke"`).

**In plain English:** slide a window over the string. Grow it on the right; when the new character is already inside the window, jump the left edge just past that character's previous position.

In [ ]:
def longest_unique_brute(s: str) -> int:
    best = 0
    for i in range(len(s)):                           # every start...
        seen = set()
        for ch in s[i:]:                              # ...extend until a repeat
            if ch in seen:
                break
            seen.add(ch)
        best = max(best, len(seen))
    return best


def longest_unique(s: str) -> tuple[int, str]:
    last_seen = {}                                    # char -> index of its latest occurrence
    start = best_start = best_len = 0
    for i, ch in enumerate(s):
        if last_seen.get(ch, -1) >= start:            # repeat INSIDE the window: move start past it
            start = last_seen[ch] + 1
        last_seen[ch] = i
        if i - start + 1 > best_len:
            best_start, best_len = start, i - start + 1
    return best_len, s[best_start:best_start + best_len]


print(longest_unique("abcabcbb"), longest_unique("pwwkew"), longest_unique("abba"))

In [ ]:
cases = {"": (0, ""), "a": (1, "a"), "bbbbb": (1, "b"), "abcabcbb": (3, "abc"),
         "pwwkew": (3, "wke"), "dvdf": (3, "vdf"), "abba": (2, "ab"), " ": (1, " "),
         "😀é😀": (2, "😀é")}
for s, expected in cases.items():
    assert longest_unique(s) == expected, s
    assert longest_unique_brute(s) == expected[0], s
for _ in range(500):                                  # property test against the brute force
    s = "".join(random.choice("abcd") for _ in range(random.randint(0, 12)))
    assert longest_unique(s)[0] == longest_unique_brute(s), s
print("longest unique substring: all tests passed")

`"abba"` is the case that breaks most first attempts: when the second `a` arrives, its previous index (0) is already *left* of the window, so `start` must not move back. The timing below uses the brute force's worst case, a string with no repeats at all.

In [ ]:
sizes = [250, 500, 1000, 2000, 3000]
brute_ms, window_ms = [], []
for n in sizes:
    s = "".join(chr(0x4E00 + i) for i in range(n))    # n distinct CJK characters
    t0 = time.perf_counter(); assert longest_unique_brute(s) == n
    brute_ms.append((time.perf_counter() - t0) * 1000)
    t0 = time.perf_counter(); assert longest_unique(s)[0] == n
    window_ms.append((time.perf_counter() - t0) * 1000)
print("brute ms :", [round(t, 1) for t in brute_ms])
print("window ms:", [round(t, 2) for t in window_ms])
assert brute_ms[-1] > 10 * window_ms[-1]

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(sizes, brute_ms, color=ORANGE, lw=2, marker="o", label="brute force: every start, extend")
ax.plot(sizes, window_ms, color=BLUE, lw=2, marker="o", label="sliding window with last-seen index")
ax.set_xlabel("string length (no repeated characters)"); ax.set_ylabel("milliseconds")
ax.set_title("O(n²) vs O(n): the gap grows with every extra character")
ax.legend(loc="upper left")
plt.show()

**Complexity:** the sliding window is O(n) time and O(min(n, alphabet)) space; the brute force is O(n²) in the worst case. The same window-plus-hash-map template solves "at most k distinct characters" and "minimum window substring".

## 4 · Encoding, ciphers and parsing

### Problem 12 — Run-length encoding and decoding

Example: `"aaabccdddd"` ↔ `"3a1b2c4d"`; counts can exceed 9: `"a" * 12` → `"12a"`. Assume the input contains no digits (clarify!).

In [ ]:
def rle_encode(s: str) -> str:
    return "".join(f"{len(list(run))}{ch}" for ch, run in groupby(s))   # groupby yields runs of equal items


def rle_encode_loop(s: str) -> str:
    if not s:
        return ""
    out, prev, count = [], s[0], 1
    for ch in s[1:]:
        if ch == prev:
            count += 1
        else:
            out.append(f"{count}{prev}")
            prev, count = ch, 1
    out.append(f"{count}{prev}")                  # flush the final run
    return "".join(out)


def rle_decode(encoded: str) -> str:
    return "".join(ch * int(n) for n, ch in re.findall(r"(\d+)(\D)", encoded))


print(rle_encode("aaabccdddd"), rle_decode("3a1b2c4d"), rle_encode("a" * 12))

In [ ]:
for enc in (rle_encode, rle_encode_loop):
    assert enc("") == "" and rle_decode("") == ""          # empty
    assert enc("a") == "1a"                                # single
    assert enc("aaabccdddd") == "3a1b2c4d"
    assert enc("a" * 12) == "12a"                          # multi-digit counts
    assert enc("ééé😀😀") == "3é2😀"                         # unicode
for _ in range(300):                                       # property test: decode(encode(s)) == s
    s = "".join(random.choice("ab ") for _ in range(random.randint(0, 15)))
    assert rle_decode(rle_encode(s)) == s and rle_encode(s) == rle_encode_loop(s)
assert rle_decode(rle_encode("11")) != "11"                # digits in the input break the format
assert len(rle_encode("abc")) > len("abc")                 # RLE only helps when there are runs
print("run-length encoding: all tests passed")

**Complexity:** O(n) time and space. Say out loud that the format is ambiguous if the input may contain digits (escape them, or emit `(char, count)` pairs), and that RLE makes text without runs longer.

### Problem 13 — Caesar cipher

Shift letters by k with wrap-around, keep case, leave everything else alone; decrypt with −k.
Example: `caesar("Hello, World!", 3)` → `"Khoor, Zruog!"`.

In [ ]:
def caesar(text: str, k: int) -> str:
    k %= 26                                            # handles negative and large shifts
    lo, up = string.ascii_lowercase, string.ascii_uppercase
    table = str.maketrans(lo + up, lo[k:] + lo[:k] + up[k:] + up[:k])
    return text.translate(table)                       # one C-level pass


def caesar_loop(text: str, k: int) -> str:
    out = []
    for ch in text:
        if "a" <= ch <= "z":
            out.append(chr((ord(ch) - ord("a") + k) % 26 + ord("a")))
        elif "A" <= ch <= "Z":
            out.append(chr((ord(ch) - ord("A") + k) % 26 + ord("A")))
        else:
            out.append(ch)
    return "".join(out)


print(caesar("Hello, World!", 3), "|", caesar(caesar("Hello, World!", 3), -3))

In [ ]:
for f in (caesar, caesar_loop):
    assert f("", 5) == ""                                  # empty
    assert f("a", 1) == "b"                                # single
    assert f("xyz", 3) == "abc"                            # wrap-around
    assert f("abc", -1) == "zab"                           # negative shift
    assert f("abc", 29) == f("abc", 3)                     # shifts are mod 26
    assert f("Hello, World!", 3) == "Khoor, Zruog!"        # case and punctuation kept
    assert f("café", 1) == "dbgé"                          # non-ASCII letters untouched
    assert f(f(TEXT, 13), 13) == TEXT                      # ROT13 is its own inverse
    assert f(f(TEXT, 7), -7) == TEXT                       # decrypt = shift back
print("caesar: all tests passed")

**Complexity:** O(n) time. `str.maketrans` + `translate` is the idiomatic version. There are only 26 keys, so the cipher falls to brute force plus letter frequencies (Try it yourself #3).

### Problem 14 — Roman numerals ↔ integers

Example: `"MCMXCIV"` → `1994` and back; valid range 1–3999.

In [ ]:
ROMAN = [(1000, "M"), (900, "CM"), (500, "D"), (400, "CD"), (100, "C"), (90, "XC"),
         (50, "L"), (40, "XL"), (10, "X"), (9, "IX"), (5, "V"), (4, "IV"), (1, "I")]
VALUES = {"I": 1, "V": 5, "X": 10, "L": 50, "C": 100, "D": 500, "M": 1000}


def int_to_roman(n: int) -> str:
    if not 1 <= n <= 3999:
        raise ValueError("Roman numerals cover 1..3999")
    out = []
    for value, symbol in ROMAN:                        # greedy: the biggest symbol that fits
        count, n = divmod(n, value)
        out.append(symbol * count)
    return "".join(out)


def roman_to_int(s: str) -> int:
    total = 0
    for i, ch in enumerate(s):
        value = VALUES[ch]
        if i + 1 < len(s) and VALUES[s[i + 1]] > value:
            total -= value                             # smaller before bigger subtracts: IV, XC, CM
        else:
            total += value
    return total


def is_valid_roman(s: str) -> bool:
    try:
        return int_to_roman(roman_to_int(s)) == s      # only the canonical spelling round-trips
    except (KeyError, ValueError):
        return False


print(roman_to_int("MCMXCIV"), int_to_roman(1994), int_to_roman(3999))

In [ ]:
pairs = {1: "I", 4: "IV", 9: "IX", 14: "XIV", 40: "XL", 90: "XC", 400: "CD", 1994: "MCMXCIV", 3999: "MMMCMXCIX"}
for n, r in pairs.items():
    assert int_to_roman(n) == r and roman_to_int(r) == n
assert all(roman_to_int(int_to_roman(n)) == n for n in range(1, 4000))   # round trip for every number
assert raises(ValueError, int_to_roman, 0) and raises(ValueError, int_to_roman, -5)
assert raises(ValueError, int_to_roman, 4000)
assert is_valid_roman("MCMXCIV")
assert not any(is_valid_roman(s) for s in ("", "IIII", "IC", "MMMM", "xiv", "ABC"))   # empty, non-canonical, bad chars
print("roman numerals: all tests passed")

**Complexity:** O(1) in practice (at most 15 symbols; the table has 13 entries). `roman_to_int` alone accepts nonsense such as `"IC"`; round-tripping through `int_to_roman` is a two-line validator.

### Problem 15 — Email and URL validation with regex

Write a pragmatic check that a string looks like an email or an http(s) URL, and explain what regex cannot do here.

**In plain English:** a regex can say "this is plausibly an email"; only sending a confirmation email proves it is real. The official grammar (RFC 5322) allows quoted spaces, IP-address domains and more, so any short regex both rejects some valid addresses and accepts some invalid ones.

In [ ]:
EMAIL_RE = re.compile(r"[A-Za-z0-9._%+-]+@(?:[A-Za-z0-9-]+\.)+[A-Za-z]{2,}")
URL_RE = re.compile(r"https?://[^\s/$.?#][^\s]*", re.IGNORECASE)     # for FINDING urls in text


def looks_like_email(s: str) -> bool:
    return EMAIL_RE.fullmatch(s) is not None          # fullmatch: nothing allowed before or after


def looks_like_url(s: str) -> bool:
    parts = urlparse(s)                               # a real parser beats a regex for validation
    return parts.scheme in {"http", "https"} and bool(parts.netloc) and not any(c.isspace() for c in s)


reply = "See https://docs.python.org/3/library/re.html and http://localhost:8000/health, or mail help@example.com."
print("urls  :", URL_RE.findall(reply))
print("clean :", [u.rstrip(".,;:!?)") for u in URL_RE.findall(reply)])   # trailing punctuation sneaks in
print("emails:", EMAIL_RE.findall(reply))

In [ ]:
valid = ["alice@example.com", "first.last+tag@sub.example.co.uk", "a_b%c@x-y.io"]
invalid = ["", "plainaddress", "@example.com", "alice@", "alice@example", "alice@example.c",
           "alice smith@example.com", "alice@exa mple.com"]
assert all(looks_like_email(e) for e in valid) and not any(looks_like_email(e) for e in invalid)
assert EMAIL_RE.match("alice@example.com; DROP TABLE") is not None   # match() anchors only the START
assert not looks_like_email("alice@example.com; DROP TABLE")         # fullmatch() anchors both ends

# Why regex is imperfect for email: it fails in BOTH directions
rfc_valid_but_rejected = ["user@localhost", '"john doe"@example.com', "josé@example.com", "user@[192.168.0.1]"]
invalid_but_accepted = ["a..b@example.com", "a@-bad-.com", ".alice@example.com"]
assert not any(looks_like_email(e) for e in rfc_valid_but_rejected)
assert all(looks_like_email(e) for e in invalid_but_accepted)

assert looks_like_url("https://example.com/path?q=1#top") and looks_like_url("http://localhost:8000")
for bad in ["", "example.com", "https://", "ftp://example.com", "javascript:alert(1)", "https://exa mple.com"]:
    assert not looks_like_url(bad), bad               # no scheme, no host, wrong scheme, spaces
print("email/url validation: all tests passed")

**Complexity:** O(n) per check; these patterns have no nested unbounded quantifiers, so there is no catastrophic backtracking (ReDoS). Allowlist URL schemes, because `javascript:` links are a classic XSS vector. In production use a maintained library (for example `email-validator`) for syntax, and a confirmation link for truth.

## Try it yourself

**1.** Can the letters of a string be rearranged into a palindrome? `"carerac"` → `True`, `"code"` → `False`. Hint: count how many letters appear an odd number of times.

In [ ]:
# Solution — try it yourself first, then run this
def can_form_palindrome(s: str) -> bool:
    odd = sum(count % 2 for count in Counter(s).values())
    return odd <= 1                                    # at most one letter sits in the middle


assert can_form_palindrome("") and can_form_palindrome("a")
assert can_form_palindrome("carerac") and can_form_palindrome("aabb")
assert not can_form_palindrome("code") and not can_form_palindrome("ab")
print({s: can_form_palindrome(s) for s in ("carerac", "aabb", "code")})

**2.** Longest palindromic substring: `"babad"` → `"bab"` (or `"aba"`), `"cbbd"` → `"bb"`. Hint: every palindrome has a centre, either one character or the gap between two, so expand outwards from each of the 2n − 1 centres.

In [ ]:
# Solution — try it yourself first, then run this
def longest_palindrome(s: str) -> str:
    best = ""
    for centre in range(2 * len(s) - 1):
        lo, hi = centre // 2, (centre + 1) // 2        # odd centres: lo == hi; even: neighbours
        while lo >= 0 and hi < len(s) and s[lo] == s[hi]:
            lo, hi = lo - 1, hi + 1
        if hi - lo - 1 > len(best):
            best = s[lo + 1:hi]
    return best


assert longest_palindrome("") == "" and longest_palindrome("a") == "a"
assert longest_palindrome("babad") in {"bab", "aba"} and longest_palindrome("cbbd") == "bb"
assert longest_palindrome("forgeeksskeegfor") == "geeksskeeg"
print(longest_palindrome("babad"), longest_palindrome("cbbd"), longest_palindrome("racecar level"))

**3.** Crack a Caesar cipher without the key. Try all 26 shifts and score each candidate against English letter frequencies with a chi-squared statistic (lower = more English-like).

In [ ]:
# Solution — try it yourself first, then run this
ENGLISH_PCT = {  # approximate share of each letter in English text, in %
    "e": 12.7, "t": 9.1, "a": 8.2, "o": 7.5, "i": 7.0, "n": 6.7, "s": 6.3, "h": 6.1, "r": 6.0,
    "d": 4.3, "l": 4.0, "c": 2.8, "u": 2.8, "m": 2.4, "w": 2.4, "f": 2.2, "g": 2.0, "y": 2.0,
    "p": 1.9, "b": 1.5, "v": 1.0, "k": 0.8, "j": 0.15, "x": 0.15, "q": 0.1, "z": 0.07,
}


def chi_squared(text: str) -> float:
    letters = [ch for ch in text.lower() if ch in ENGLISH_PCT]
    n = len(letters) or 1
    counts = Counter(letters)
    return sum((counts[ch] - n * p / 100) ** 2 / (n * p / 100) for ch, p in ENGLISH_PCT.items())


def crack_caesar(ciphertext: str):
    scores = [chi_squared(caesar(ciphertext, -k)) for k in range(26)]
    key = min(range(26), key=scores.__getitem__)
    return key, caesar(ciphertext, -key), scores


secret = caesar(TEXT, 11)
key, plain, scores = crack_caesar(secret)
print(f"recovered key = {key}; first words: {plain[:40]!r}")
assert key == 11 and plain == TEXT

fig, ax = plt.subplots(figsize=(9, 3.4))
ax.bar(range(26), scores, color=[BLUE if k == key else GREY for k in range(26)], width=0.7)
ax.set_yscale("log")
ax.set_xlabel("shift tried"); ax.set_ylabel("chi-squared (log scale)")
ax.set_title(f"Only shift {key} reads like English: its score is far below every other shift")
plt.show()

## Say it in an interview

**30-second answer for "how do you approach string problems?":** *"I first ask about the alphabet (ASCII or unicode, case-sensitive or not) and about empty input. Then I pick a pattern: a counter for frequency questions, a stack for nesting, a sliding window with a last-seen map for substring questions, a lookup table for symbol mapping. Most of these are O(n) time with O(k) extra space for k distinct characters."*

| Pattern | Problems here | Complexity |
|---|---|---|
| Counter / hash map | first unique, top words, isomorphic, can-form-palindrome | O(n) time, O(k) space |
| Stack | valid brackets | O(n) time and space |
| Sliding window + last-seen index | longest substring without repeats | O(n) time |
| Expand around centre | longest palindromic substring | O(n²) time, O(1) space |
| Substring trick | rotation: `b in a + a` (with a length check) | O(n) |
| Lookup table / greedy | Roman numerals, Caesar via `str.maketrans` | O(n) |

**Traps interviewers probe:** `split(" ")` vs `split()`; `str.title()` breaks on apostrophes; `Counter.most_common` breaks ties by first appearance; `re.match` anchors only the start (use `fullmatch`); `"é"` can be one code point or two (normalise); stripping accents is lossy (й → и); regex email validation fails both ways; allowlist URL schemes.

**Follow-ups:** "What if the text is a stream?" (keep counts and window state incrementally). "What about unicode?" (code points vs grapheme clusters; `casefold`; NFC/NFKD). "Why is `s += ch` in a loop risky?" (strings are immutable; each `+` can copy, O(n²) in the worst case; build a list and `"".join` it).

## Next

- [03 · List and dict problems](03_list_and_dict_problems.ipynb): hashing, two pointers, counting on lists and dicts.
- [04 · Practical AI-engineer Python](04_practical_ai_engineer_python.ipynb): the same skills on LLM output, logs and prompts.
- [Hashing and sliding window](../03_dsa/03_hashing_and_sliding_window.ipynb) and [Stacks and queues](../03_dsa/04_stacks_and_queues.ipynb): the patterns behind problems 5–11.
- [Tokenization, BPE and tiktoken](../07_genai_foundations/01_tokenization_bpe_and_tiktoken.ipynb): how models split text (not into words).
- [BM25 from scratch](../09_rag/02_bm25_from_scratch.ipynb): tokenising and normalising text for keyword search.
- Theory: [python_basics course](../../python_basics/index.html) · [dsa_basics course](../../dsa_basics/index.html)